# M16 — Use Matrices as Transformations

## Begin visually

The same arrow appears twice below. One copy has been transformed by a small table of numbers. Before naming the operation, predict which landmarks move right, left, up, or down. Look for what stays straight and whether parallel edges remain parallel.

In [ ]:
from pathlib import Path
import csv
import math
import numpy as np
import matplotlib
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

def locate(relative):
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / relative
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative)

DATA_PATH = locate(Path("datasets/M16/shape_points.csv"))
with DATA_PATH.open(encoding="utf-8", newline="") as handle:
    rows = sorted(csv.DictReader(handle), key=lambda row: int(row["vertex_order"]))
POINTS = np.array([[float(row["x"]), float(row["y"])] for row in rows])
POINT_IDS = [row["point_id"] for row in rows]

def plot_shape(ax, points, label, color):
    closed = np.vstack([points, points[0]])
    ax.plot(closed[:, 0], closed[:, 1], marker="o", color=color, label=label)
    ax.axhline(0, color="0.85", linewidth=1)
    ax.axvline(0, color="0.85", linewidth=1)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.2)

MYSTERY = np.array([[1.25, 0.65], [-0.15, 0.85]])
MYSTERY_POINTS = POINTS @ MYSTERY.T
fig, ax = plt.subplots(figsize=(8, 5))
plot_shape(ax, POINTS, "original", "#386cb0")
plot_shape(ax, MYSTERY_POINTS, "transformed", "#ef6c35")
ax.set_title("What did the table of numbers do?")
ax.legend()
plt.show()
assert POINTS.shape == MYSTERY_POINTS.shape == (7, 2)

Record the movement of the `tip` and `neck_upper` landmarks before continuing. A matrix transformation can scale, rotate, shear, reflect, project, or combine these effects while mapping every input vector by the same rule.

## Matrix × vector: build each output coordinate

**Prediction checkpoint — write before running:** for `x = [2, -1]` and the matrix below, compute the two row dot products. Predict the output shape as well as the values. This mission uses the single-vector column convention `y = A @ x`.

In [ ]:
A = np.array([[2.0, 1.0], [-1.0, 3.0]])
x = np.array([2.0, -1.0])
row_calculation = np.array([2.0 * 2.0 + 1.0 * -1.0, -1.0 * 2.0 + 3.0 * -1.0])
y = A @ x
print("A shape:", A.shape, "x shape:", x.shape, "y shape:", y.shape)
print("row calculations:", row_calculation, "A @ x:", y)
assert np.allclose(y, row_calculation)
assert y.shape == (2,)

The columns of `A` show where the basis vectors go: `A @ [1, 0]` is the first column and `A @ [0, 1]` is the second. Any input combines those transformed basis vectors using its original coordinates.

In [ ]:
e1 = np.array([1.0, 0.0])
e2 = np.array([0.0, 1.0])
print("A e1:", A @ e1, "first column:", A[:, 0])
print("A e2:", A @ e2, "second column:", A[:, 1])
assert np.allclose(A @ e1, A[:, 0])
assert np.allclose(A @ e2, A[:, 1])

## Scaling

**Prediction checkpoint — write before running:** if horizontal coordinates double while vertical coordinates halve, where will the arrow tip `[2, 0]` and upper neck `[0.5, 0.45]` move? Predict whether area increases, decreases, or stays equal.

In [ ]:
SCALE = np.array([[2.0, 0.0], [0.0, 0.5]])
scaled = POINTS @ SCALE.T
fig, ax = plt.subplots(figsize=(8, 5))
plot_shape(ax, POINTS, "original", "#386cb0")
plot_shape(ax, scaled, "scaled", "#7b3294")
ax.set_title("Non-uniform scaling")
ax.legend()
plt.show()
print("tip:", SCALE @ np.array([2.0, 0.0]))
print("area multiplier (determinant):", np.linalg.det(SCALE))
assert np.allclose(SCALE @ np.array([2.0, 0.0]), [4.0, 0.0])
assert np.isclose(np.linalg.det(SCALE), 1.0)

## Rotation

**Prediction checkpoint — write before running:** on Cartesian axes, predict where `[1, 0]` and `[0, 1]` land after a `60°` counter-clockwise rotation. Also predict whether lengths and area change.

In [ ]:
def rotation(degrees):
    theta = math.radians(degrees)
    c, s = math.cos(theta), math.sin(theta)
    return np.array([[c, -s], [s, c]])

ROTATE_60 = rotation(60)
rotated = POINTS @ ROTATE_60.T
fig, ax = plt.subplots(figsize=(7, 6))
plot_shape(ax, POINTS, "original", "#386cb0")
plot_shape(ax, rotated, "rotated 60°", "#1b9e77")
ax.set_title("Rotation preserves lengths and area")
ax.legend()
plt.show()
print("R e1:", ROTATE_60 @ e1, "R e2:", ROTATE_60 @ e2)
assert np.allclose(np.linalg.norm(rotated, axis=1), np.linalg.norm(POINTS, axis=1))
assert np.isclose(np.linalg.det(ROTATE_60), 1.0)

## Shearing

**Prediction checkpoint — write before running:** for horizontal shear `x' = x + 0.75y`, predict which points do not move and whether the upper edge moves left or right. Then predict the corresponding vertical shear.

In [ ]:
SHEAR_X = np.array([[1.0, 0.75], [0.0, 1.0]])
SHEAR_Y = np.array([[1.0, 0.0], [-0.4, 1.0]])
sheared_x = POINTS @ SHEAR_X.T
sheared_y = POINTS @ SHEAR_Y.T
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, transformed, title, color in [
    (axes[0], sheared_x, "horizontal shear", "#d95f02"),
    (axes[1], sheared_y, "vertical shear", "#7570b3"),
]:
    plot_shape(ax, POINTS, "original", "#386cb0")
    plot_shape(ax, transformed, title, color)
    ax.set_title(title)
    ax.legend()
plt.show()
assert np.allclose(SHEAR_X @ np.array([2.0, 0.0]), [2.0, 0.0])
assert np.isclose(np.linalg.det(SHEAR_X), 1.0)

## Composition and order

**Prediction checkpoint — write before running:** start with landmark `p = [1, 0]`. First apply horizontal shear, then rotate `90°`. Compute both intermediate vectors. In the column convention, the composite is `ROTATE_90 @ SHEAR_X`: the rightmost matrix acts first.

In [ ]:
ROTATE_90 = rotation(90)
p = np.array([1.0, 0.0])
after_shear = SHEAR_X @ p
after_rotate = ROTATE_90 @ after_shear
CORRECT_COMPOSITE = ROTATE_90 @ SHEAR_X
one_step = CORRECT_COMPOSITE @ p
print("p -> shear -> rotate:", p, after_shear, after_rotate)
print("one composite step:", one_step)
assert np.allclose(one_step, after_rotate)
assert np.allclose(POINTS @ CORRECT_COMPOSITE.T, (POINTS @ SHEAR_X.T) @ ROTATE_90.T)

**Prediction checkpoint — write before running:** will `rotate then shear` produce the same arrow as `shear then rotate`? Name one landmark likely to expose the difference. Matrix multiplication is associative, but generally not commutative.

In [ ]:
shear_then_rotate = POINTS @ (ROTATE_90 @ SHEAR_X).T
rotate_then_shear = POINTS @ (SHEAR_X @ ROTATE_90).T
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
plot_shape(axes[0], shear_then_rotate, "shear then rotate", "#1b9e77")
plot_shape(axes[1], rotate_then_shear, "rotate then shear", "#e7298a")
axes[0].legend(); axes[1].legend()
plt.show()
print("largest coordinate difference:", np.max(np.abs(shear_then_rotate - rotate_then_shear)))
assert not np.allclose(ROTATE_90 @ SHEAR_X, SHEAR_X @ ROTATE_90)

## Controlled failure A — plausible, wrong order

Intent: shear, then rotate. The seeded composite below reverses application order.

**Prediction checkpoint — write before running:** both results have shape `(7, 2)` and both look like transformed arrows. Predict the smallest landmark input that will distinguish them and the first intermediate value that should differ.

In [ ]:
WRONG_ORDER_COMPOSITE = SHEAR_X @ ROTATE_90  # seeded root cause
wrong_order_output = POINTS @ WRONG_ORDER_COMPOSITE.T
intended_output = POINTS @ CORRECT_COMPOSITE.T
landmark = np.array([0.0, 1.0])
wrong_landmark = WRONG_ORDER_COMPOSITE @ landmark
intended_landmark = ROTATE_90 @ (SHEAR_X @ landmark)
print("same outer shape:", wrong_order_output.shape == intended_output.shape)
print("landmark wrong/intended:", wrong_landmark, intended_landmark)
print("invariant holds:", np.allclose(wrong_order_output, intended_output))
assert wrong_order_output.shape == intended_output.shape
assert not np.allclose(wrong_landmark, intended_landmark)

Repair the cause, not the picture: encode the verbal pipeline, then require the one-step composite to equal the explicit two-step computation for every row.

In [ ]:
repaired_order_output = POINTS @ (ROTATE_90 @ SHEAR_X).T
explicit_order_output = (POINTS @ SHEAR_X.T) @ ROTATE_90.T
assert np.allclose(repaired_order_output, explicit_order_output)
print("repaired order satisfies explicit-step invariant")

## Batching and why transpose is useful

For one column vector, `y = A @ x`. A NumPy batch stores samples as rows: `X.shape == (batch, features)`. Transposing the single-vector equation gives `y.T = x.T @ A.T`, so the whole batch is `Y = X @ A.T`.

**Prediction checkpoint — write before running:** the seeded `X @ A` below still returns `(7, 2)` because `A` is square. Predict why its numbers can nevertheless represent the wrong transformation.

In [ ]:
ASYMMETRIC_A = np.array([[1.0, 0.8], [-0.25, 1.0]])
correct_batch = POINTS @ ASYMMETRIC_A.T
wrong_orientation_batch = POINTS @ ASYMMETRIC_A  # seeded root cause
print("X, A, correct, wrong shapes:", POINTS.shape, ASYMMETRIC_A.shape, correct_batch.shape, wrong_orientation_batch.shape)
print("tip correct/wrong:", correct_batch[POINT_IDS.index("tip")], wrong_orientation_batch[POINT_IDS.index("tip")])
assert correct_batch.shape == wrong_orientation_batch.shape == (7, 2)
assert not np.allclose(correct_batch, wrong_orientation_batch)

Use sample-wise equivalence as the repair invariant: batch row `i` must equal `A @ X[i]` under the declared column convention. This is stronger than checking only the outer shape.

In [ ]:
samplewise = np.stack([ASYMMETRIC_A @ point for point in POINTS])
repaired_batch = POINTS @ ASYMMETRIC_A.T
assert np.allclose(repaired_batch, samplewise)
print("batch equals stacked single-vector transformations:", np.allclose(repaired_batch, samplewise))

## Shape reasoning before multiplication

For `(m, n) @ (n, p)`, the inner dimensions must agree and the output is `(m, p)`. Shapes can reject impossible products, but matching shapes do not prove semantic intent.

**Prediction checkpoint — write before running:** for `X: (7, 2)` and `B: (3, 2)`, predict whether `X @ B`, `X @ B.T`, and `B @ X.T` run, and give each valid output shape.

In [ ]:
B = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, -1.0]])  # three outputs from two inputs
shape_results = {
    "X @ B.T": (POINTS @ B.T).shape,
    "B @ X.T": (B @ POINTS.T).shape,
}
try:
    POINTS @ B
except ValueError as error:
    hard_dimension_failure = str(error)
else:
    raise AssertionError("X @ B should fail because 2 != 3")
print(shape_results)
print("caught incompatible inner dimensions:", hard_dimension_failure.splitlines()[0])
assert shape_results == {"X @ B.T": (7, 3), "B @ X.T": (3, 7)}

## Batched computation

**Prediction checkpoint — write before running:** predict whether transforming 2,800 points in one matrix multiplication changes the numeric result relative to a Python loop. The batch operation expresses the same independent sample rule compactly.

In [ ]:
LARGE_BATCH = np.tile(POINTS, (400, 1))
vectorized = LARGE_BATCH @ MYSTERY.T
looped = np.stack([MYSTERY @ point for point in LARGE_BATCH])
print("batch shape:", LARGE_BATCH.shape, "output shape:", vectorized.shape)
assert LARGE_BATCH.shape == (2800, 2)
assert np.allclose(vectorized, looped)

## Connection to ML batches and layers

A dense layer receives `X: (batch, in_features)`, multiplies by `W: (in_features, out_features)`, adds `b: (out_features,)` by broadcasting, and may apply a nonlinearity.

**Prediction checkpoint — write before running:** determine every intermediate shape below. Then calculate the first sample independently and predict whether it equals row zero of the batched result.

In [ ]:
X_BATCH = np.array([[1.0, 0.0, -1.0], [0.5, 2.0, 1.0], [-1.0, 1.0, 0.0], [2.0, -0.5, 0.5]])
WEIGHTS = np.array([[0.5, -1.0], [1.0, 0.25], [-0.5, 2.0]])
BIAS = np.array([0.1, -0.2])
pre_activation = X_BATCH @ WEIGHTS + BIAS
relu_output = np.maximum(pre_activation, 0.0)
first_sample = X_BATCH[0] @ WEIGHTS + BIAS
print("X @ W + b shapes:", X_BATCH.shape, WEIGHTS.shape, BIAS.shape, pre_activation.shape)
print("first sample:", first_sample, "batch row zero:", pre_activation[0])
assert pre_activation.shape == (4, 2)
assert np.allclose(first_sample, pre_activation[0])
assert np.all(relu_output >= 0)

For the same layer viewed one column vector at a time, define `A = W.T`, shaped `(out_features, in_features)`. Then `A @ x + b` equals the corresponding row of `X @ W + b`. The apparent formula difference comes from representation, not different computation.

In [ ]:
A_COLUMN_VIEW = WEIGHTS.T
column_view_outputs = np.stack([A_COLUMN_VIEW @ sample + BIAS for sample in X_BATCH])
assert A_COLUMN_VIEW.shape == (2, 3)
assert np.allclose(column_view_outputs, pre_activation)
print("row-batch and column-vector layer views agree")

## Code reading and transfer

Before running the next cell, complete the trace in `missions/M16/code_reading.md`: inputs, axis meanings, mutation, loop order, intermediate landmark values, and output list length.

In [ ]:
def apply_layers(points, matrices):
    states = [points.copy()]
    for matrix in matrices:
        if matrix.ndim != 2 or points.shape[1] != matrix.shape[1]:
            raise ValueError(f"cannot map {points.shape} with {matrix.shape}")
        points = points @ matrix.T
        states.append(points.copy())
    return states

states = apply_layers(POINTS, [SHEAR_X, ROTATE_90])
assert len(states) == 3
assert np.allclose(states[-1], explicit_order_output)
assert not np.shares_memory(states[0], POINTS)
print([state.shape for state in states])

## No-AI Gate and ADR

Close this notebook and complete `missions/M16/no_ai_gate.md` without AI-generated code. Then write the representation decision requested by `missions/M16/adr_prompt.md` using `templates/ADR.md`.

Your evidence must include predictions made before action, both plausible-failure diagnoses, an explicit shape table, sample-wise batch equivalence, and a plain-language ML-layer connection. Do not treat a successful run or plausible plot as proof of semantic correctness.